# EMA confirmatory run: Qwen3-4B, conditions A–E

Runs [`ema.run_confirmatory`](../src/ema/run_confirmatory.py): 5 memory conditions × (103 `oneoff` tasks × 3 epochs + 103 `original` tasks × 1 epoch), with the settings frozen in the README.

The run takes several Kaggle sessions. Each session:
1. restores the logs of earlier sessions from an attached dataset, if there is one;
2. resumes the eval set, so finished samples are reused;
3. stops gracefully before Kaggle's 12-hour limit, so its logs are saved as output.

Upload this session's `logs/` as the next version of the dataset, then run again until it prints `EVAL SET COMPLETE`.

**Kaggle setup:** Accelerator *GPU T4 x2*, Internet *on*. Model-written code runs on this VM (`local` sandbox).

In [ ]:
import os, glob, shutil, signal, subprocess, sys, time, torch
START = time.time()
REPO = "https://github.com/NischalParulekar/EMA"
WORK = "/kaggle/working"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
TP = max(1, torch.cuda.device_count())
LOG_DIR = f"{WORK}/logs/confirmatory_qwen3-4b"
DEADLINE_HOURS = 11.0   # stop the eval this long after the session started
PY = sys.executable
os.makedirs(LOG_DIR, exist_ok=True)
os.chdir(WORK)
print("GPUs:", TP)

## 1. Restore logs from earlier sessions

In [ ]:
found = glob.glob("/kaggle/input/**/eval-set.json", recursive=True)
if found:
    src = os.path.dirname(found[0])
    for f in os.listdir(src):
        shutil.copy2(os.path.join(src, f), LOG_DIR)
    print(f"restored {len(os.listdir(LOG_DIR))} files from {src}")
else:
    print("no earlier logs attached: starting fresh")

## 2. Environment

In [ ]:
%%bash -s "$REPO" "$WORK" "$PY"
set -e
cd "$2"
[ -d EMA ] || git clone -q "$1" EMA
pip install -q uv
# Inspect's OpenAI-compatible client needs openai>=3.4; the Kaggle image ships 2.x.
uv pip install -q --python "$3" vllm "openai>=3.4" -e "./EMA[dev]"
uv pip install -q --python "$3" --no-deps -r EMA/requirements-impossiblebench.txt
git -C EMA rev-parse HEAD | tee commit.txt
"$3" --version
uv pip freeze --python "$3" > env.txt
grep -iE "^(vllm|inspect-ai|openai|torch|transformers|impossiblebench)" env.txt

## 3. Serve the model with vLLM
T4s have no bf16, so `--dtype half`.

In [ ]:
VLLM = shutil.which("vllm") or os.path.join(os.path.dirname(PY), "vllm")
server = subprocess.Popen(
    [VLLM, "serve", MODEL, "--dtype", "half", "--tensor-parallel-size", str(TP),
     "--max-model-len", "32768", "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open(f"{WORK}/vllm.log", "w"), stderr=subprocess.STDOUT,
)
import urllib.request
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2); print("vLLM ready"); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError(open(f"{WORK}/vllm.log").read()[-3000:])
        time.sleep(10)
else:
    raise TimeoutError("vLLM did not come up; see vllm.log")

## 4. Run (resumes), stopping gracefully at the deadline

In [ ]:
run_log = open(f"{WORK}/run.log", "a")
proc = subprocess.Popen(
    [PY, "-m", "ema.run_confirmatory", LOG_DIR, f"openai-api/vllm/{MODEL}"],
    stdout=run_log, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONIOENCODING": "utf-8"},
)
remaining = START + DEADLINE_HOURS * 3600 - time.time()
try:
    proc.wait(timeout=max(60, remaining))
except subprocess.TimeoutExpired:
    print("deadline reached: stopping the eval gracefully")
    proc.send_signal(signal.SIGINT)
    try:
        proc.wait(timeout=1200)
    except subprocess.TimeoutExpired:
        proc.kill()
run_log.close()
tail = open(f"{WORK}/run.log", encoding="utf-8", errors="replace").read().splitlines()[-25:]
print("
".join(tail))
COMPLETE = any("EVAL SET COMPLETE" in line for line in tail)
open(f"{WORK}/STATUS.txt", "w").write("COMPLETE
" if COMPLETE else "INCOMPLETE
")
print("STATUS:", "COMPLETE" if COMPLETE else "INCOMPLETE")

## 5. Progress so far

In [ ]:
!{PY} -m ema.analysis {LOG_DIR} {WORK}/summary.csv
!{PY} -m ema.stats {LOG_DIR} oneoff
server.terminate()